# Initialization

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read bronze table

In [0]:
df = spark.table("bikescatalog.bronze.erp_cust_az12")

In [0]:
df.display()

# Silver Transformation

# Trimmming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Customer ID Cleanup

In [0]:
df = df.withColumn(
        "CID",
            f.when(
                    col("CID").startswith("NAS"),
                     f.substring(col("CID"), 4, f.length(col("CID"))))
            .otherwise(col("CID"))
         )   

# Birthdate Validation

In [0]:
df = df.withColumn("BDATE",
                   f.when(col("BDATE") > f.current_date(), None)
                   .otherwise(col("BDATE"))
        )

# Gender Normalization

In [0]:
df = df.withColumn("gen",
                   f.when(f.upper(col("gen")).isin("F","FEMALE"), "Female")
                    .when(f.upper(col("gen")).isin("M","MALE"), "Male")
                    .otherwise("n/a")
                   )

# Rename Columns

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Sanitary check of dataframe

In [0]:
df.limit(10).display()

# writing silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("bikescatalog.silver.erp_customer")

# Sanity check of silver table

In [0]:
%sql
select * from bikescatalog.silver.erp_customer limit 10